In [1]:
# !pip install mljar-supervised
# from supervised import AutoML
# !pip install autogluon
# from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import xgboost as xgb
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns 
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier,ExtraTreesClassifier,GradientBoostingClassifier,HistGradientBoostingClassifier,AdaBoostClassifier
from sklearn.ensemble import VotingClassifier,StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# from sklearn.metrics import (    accuracy_score,    confusion_matrix,f1_score,precision_score, recall_score,roc_auc_score)
# from sklearn.metrics import make_scorer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold,cross_val_score, StratifiedShuffleSplit
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, LabelEncoder,FunctionTransformer, StandardScaler
from category_encoders import TargetEncoder, CatBoostEncoder
from category_encoders.binary import BinaryEncoder
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold, SelectFromModel,SelectPercentile
from statsmodels.stats.outliers_influence import variance_inflation_factor
pd.set_option('display.max_rows', 500)
import os
import warnings
warnings.filterwarnings('ignore')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/life-resorce/sample_submission.csv
/kaggle/input/life-resorce/train.csv
/kaggle/input/life-resorce/test.csv


In [2]:
def preprocess(data):
    # 모든 값이 동일한 열 찾기
    constant_columns = []
    for col in data.columns:
        if data[col].nunique() == 1 and data[col].isnull().sum() == 0:
            constant_columns.append(col)

    # 모든 값이 결측인 열 찾기
    no_values = []
    for col in data.columns:
        if data[col].isnull().all():
            no_values.append(col)

    columns_to_remove = constant_columns + no_values 

    # data에서 해당 열 제거
    data = data.drop(columns=columns_to_remove)
    return data
class FeatureSelect:
    """
        변수 선택 기능 
        kbest, vif, variance, model_importance 
    """
    def __init__(self, x: pd.DataFrame, y: pd.Series, test: pd.DataFrame, ratio: float, 
                 vif_threshold: float, variance_threshold: float, model=None):
        self.x = x
        self.y = y
        self.test = test
        self.ratio = ratio
        self.vif_threshold = vif_threshold
        self.var_threshold = variance_threshold
        self.model = model
        
        # 선택할 피처의 수
        self.select_nums = max(1, int(len(x.columns) * ratio))  # 최소 1개 피처 선택
        self.kbest_selector = SelectKBest(f_classif, k=self.select_nums)
        self.variance_selector = VarianceThreshold(threshold=self.var_threshold)

    def _check_data(self):
        """데이터 유효성 검사"""
        if self.x.isnull().values.any() or self.test.isnull().values.any():
            raise ValueError("Input data contains null values.")
        if self.x.shape[0] != self.y.shape[0]:
            raise ValueError("The number of samples in x and y must be equal.")

    def kbest_select(self) -> tuple:
        """SelectKBest 피처 선택"""
        self._check_data()
        self.kbest_selector.fit(self.x, self.y)
        kbest_columns = self.kbest_selector.get_support(indices=True)
        self.x = self.x.iloc[:, kbest_columns]
        self.test = self.test.iloc[:, kbest_columns]
        print("KBest selection completed.")
        return self.x, self.test

    def vif_select(self) -> tuple:
        """VIF 기반 피처 선택"""
        self._check_data()

        # VIF 계산
        vif = pd.DataFrame()
        vif['Features'] = self.x.columns
        vif['VIF'] = [
            variance_inflation_factor(self.x.values, i) for i in tqdm(range(self.x.shape[1]), desc="Calculating VIF")
        ]
        vif['VIF'] = round(vif['VIF'], 2)
        vif = vif.sort_values(by="VIF", ascending=False)
        print("VIF values (sorted):")
        print(vif)

        # VIF 기준으로 제거할 피처 찾기
        features_to_remove = vif[vif['VIF'] > self.vif_threshold]['Features'].tolist()
        x_selected = self.x.drop(columns=features_to_remove)
        test_selected = self.test.drop(columns=features_to_remove)

        if features_to_remove:            print(f"Removed features with VIF > {self.vif_threshold}: {features_to_remove}")
        else:            print("No features removed based on VIF.")
            
        # VIF 시각화
        plt.figure(figsize=(10, 6))
        plt.barh(vif['Features'], vif['VIF'], color='skyblue')
        plt.axvline(x=self.vif_threshold, color='red', linestyle='--', label=f'VIF Threshold = {self.vif_threshold}')
        plt.title("VIF Values for Features")
        plt.xlabel("VIF")
        plt.ylabel("Features")
        plt.legend()
        plt.show()
        return x_selected, test_selected
    
    def variancethreshold_select(self) -> tuple:
        """VarianceThreshold 피처 선택"""
        self.variance_selector.fit(self.x)
        selected_cols = self.variance_selector.get_support(indices=True)
        self.x = self.x.iloc[:, selected_cols]
        self.test = self.test.iloc[:, selected_cols]
        print("VarianceThreshold selection completed.")
        return self.x, self.test

    def model_select(self, model_threshold: float = None) -> tuple:
        """모델 기반 피처 선택"""
        if self.model is None:          raise ValueError("No model provided for feature selection.")
            
        # 모델에 기반한 피처 선택
        model_selector = SelectFromModel(self.model, threshold = model_threshold)
        model_selector.fit(self.x, self.y)
        # 선택된 피처의 마스크
        selected_features_mask = model_selector.get_support()
        # 선택된 피처와 삭제된 피처 계산
        selected_features = self.x.columns[selected_features_mask]
        removed_features = self.x.columns[~selected_features_mask]
        x_selected = self.x.loc[:, selected_features_mask]
        test_selected = self.test.loc[:, selected_features_mask]

        # 삭제된 피처 정보 출력
        if removed_features.any():
            print(f"Removed features: {removed_features.tolist()}")
            print(f"Number of features removed: {len(removed_features)}")
        else: print("No features removed.")

        print("Model-based feature selection completed.")

        return x_selected, test_selected

    def model_feature_importance(self):
        """모델 피처 중요도 출력 및 시각화"""
        if self.model is None:
            print("No model is fitted yet.")
            return

        # 모델 학습
        self.model.fit(self.x, self.y)
        importances = self.model.feature_importances_

        # 피처 수 확인
        if len(importances) == 0:
            print("No feature importances found.")
            return

        # 피처 중요도 정렬
        indices = np.argsort(importances)[::-1]
        print("Feature ranking:")
        for f in range(len(indices)):print(f"{f + 1}. feature {self.x.columns[indices[f]]} ({importances[indices[f]]:.6f})")

        # 중요도 시각화
        plt.figure(figsize=(20, 10))
        plt.title("Feature Importances")
        plt.bar(range(len(importances)), importances[indices], align='center')
        plt.xticks(range(len(importances)), self.x.columns[indices], rotation=90)
        plt.xlim([-1, len(importances)])
        plt.ylabel("Importance")
        plt.xlabel("Features")
        plt.tight_layout()
        plt.show()
        
class DataEncoder:
    """
    Binary, Onehot, Label, Target, KfoldTarget, Catboost 
    """
    def __init__(self, train, test, target, binary_cols=None, onehot_cols=None, catboost_cols=None, label_cols=None, base_target_cols=None, target_cols=None):
        self.train = train.copy()
        self.test = test.copy()
        self.target = target
        self.binary_cols = binary_cols
        self.onehot_cols = onehot_cols
        self.label_cols = label_cols
        self.target_cols = target_cols
        self.catboost_cols = catboost_cols
        self.base_target_cols = base_target_cols

    def onehot_encode(self):
        if self.onehot_cols is None:
            return self
        onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        tmp = pd.DataFrame(
            onehot_encoder.fit_transform(self.train[self.onehot_cols]),
            columns=onehot_encoder.get_feature_names_out()
        )
        self.train = pd.concat([self.train, tmp], axis=1).drop(columns=self.onehot_cols)
        tmp = pd.DataFrame(
            onehot_encoder.transform(self.test[self.onehot_cols]),
            columns=onehot_encoder.get_feature_names_out()
        )
        self.test = pd.concat([self.test, tmp], axis=1).drop(columns=self.onehot_cols)
        return self

    def label_encode(self):
        if self.label_cols is None:            return self
        for col in self.label_cols:
            le = LabelEncoder()
            le.fit(self.train[col])
            self.train[col] = le.transform(self.train[col])

            # 새로운 라벨 처리를 위한 딕셔너리 생성
            le_dict = dict(zip(le.classes_, le.transform(le.classes_)))

            # 테스트 데이터에 대해 라벨 변환
            self.test[col] = self.test[col].apply(lambda x: le_dict.get(x, -1))  # 새로운 값에 대해 -1로 변환

        return self

    def target_encode(self, smoothing=1):
        if self.base_target_cols is None:
            return self
        encoder = TargetEncoder(smoothing=smoothing)
        for col in self.base_target_cols:
            self.train[col] = encoder.fit_transform(self.train[col], self.train[self.target])
            self.test[col] = encoder.transform(self.test[col])
        return self

    def kfold_target_encode(self, n_splits=5, smoothing=1, min_samples_leaf=1, remove_original_col=True):
        if self.target_cols is None:
            return self
        for col in self.target_cols:
            target = self.train[self.target].values
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            oof = np.zeros(len(self.train))
            for tr_ind, val_ind in kf.split(self.train):
                x_tr, x_val = self.train.iloc[tr_ind], self.train.iloc[val_ind]
                means = x_val[col].map(x_tr.groupby(col)[self.target].mean())
                oof[val_ind] = means
            smoothing_factor = 1 / (1 + np.exp(-(oof - min_samples_leaf) / smoothing))
            oof_smooth = smoothing_factor * oof + (1 - smoothing_factor) * target.mean()
            self.train[col + '_TE'] = oof_smooth
            self.test[col + '_TE'] = self.test[col].map(self.train.groupby(col)[self.target].mean())
            if remove_original_col:
                self.train.drop(columns=[col], inplace=True)
                self.test.drop(columns=[col], inplace=True)
        return self

    def catboost_encode(self):
        if self.catboost_cols is None:
            return self
        enc = CatBoostEncoder(cols=self.catboost_cols)
        enc.fit(self.train[self.catboost_cols], self.train[self.target])
        self.train[self.catboost_cols] = enc.transform(self.train[self.catboost_cols])
        self.test[self.catboost_cols] = enc.transform(self.test[self.catboost_cols])
        return self

    def binary_encode(self):
        if self.binary_cols is None:
            return self
        additional_cols = [col for col in self.train.columns if col not in self.test.columns]
        train_temp = self.train.drop(columns=additional_cols)
        enc = BinaryEncoder(cols=self.binary_cols)
        train_enc = enc.fit_transform(train_temp)
        test_enc = enc.transform(self.test)
        for col in additional_cols:
            train_enc[col] = self.train[col]
        self.train = train_enc
        self.test = test_enc
        return self

In [3]:
train = pd.read_csv("/kaggle/input/life-resorce/train.csv").drop(columns = ['ID'])
test = pd.read_csv("/kaggle/input/life-resorce/test.csv").drop(columns = ['ID'])


In [4]:
# SUBCLASS 가 범주형이기 때문에 LabelEncoder 사용
le_subclass = LabelEncoder()
train['SUBCLASS'] = le_subclass.fit_transform(train['SUBCLASS'])
print("SUBCLASS 인코딩 완료 ")
# encoding 
non_num_cols = train.select_dtypes(exclude='number').columns
enc = DataEncoder(train=train,test =test ,target='SUBCLASS',base_target_cols=non_num_cols)
data =enc.target_encode()
train,test =data.train,data.test 
x_train = train.drop(columns=['SUBCLASS'])
y_subclass = train['SUBCLASS'] 
x_test = test.copy()

SUBCLASS 인코딩 완료 


In [5]:
selector= FeatureSelect(x=x_train,y=y_subclass,
                         test = x_test, ratio = 0.8 ,vif_threshold = 10, variance_threshold = 0.05,
                         model= LGBMClassifier(random_state=42,verbose =-1 , ))
# selector.model_feature_importance()
fs_train,fs_test=selector.model_select(model_threshold=3)

Removed features: ['AADAT', 'AARS1', 'ABAT', 'ABCB6', 'ABCB7', 'ABCB8', 'ABCB9', 'ABCG2', 'ABHD2', 'ABI1', 'ACAA1', 'ACAA2', 'ACADL', 'ACAT2', 'ACKR1', 'ACO2', 'ACOT2', 'ACOT8', 'ACOX1', 'ACOX3', 'ACP1', 'ACP2', 'ACP5', 'ACRBP', 'ACRV1', 'ACSL3', 'ACSL5', 'ACSM1', 'ACTA1', 'ACTA2', 'ACTC1', 'ACTG1', 'ACTN2', 'ACTN3', 'ACTR2', 'ACTR3', 'ACVRL1', 'ADA', 'ADAD1', 'ADAM8', 'ADCYAP1', 'ADGRA2', 'ADGRE1', 'ADGRG1', 'ADGRL2', 'ADGRL4', 'ADH1C', 'ADH5', 'ADH7', 'ADIG', 'ADIPOR1', 'ADIPOR2', 'ADM', 'ADORA2B', 'ADRA1B', 'ADRA2B', 'ADRA2C', 'AEBP1', 'AGER', 'AGPAT3', 'AGR2', 'AHSP', 'AIMP2', 'AK1', 'AK3', 'AK4', 'AKAP7', 'AKR1A1', 'AKR1B10', 'AKR1C2', 'AKR1D1', 'AKT1S1', 'ALAD', 'ALDH1A2', 'ALDH2', 'ALDH3A2', 'ALDH3B1', 'ALDH6A1', 'ALDH9A1', 'ALYREF', 'AMD1', 'AMH', 'AMIGO2', 'AMMECR1', 'ANG', 'ANGPTL3', 'ANGPTL4', 'ANKRA2', 'ANKZF1', 'ANLN', 'ANP32E', 'ANXA10', 'ANXA13', 'ANXA2', 'ANXA4', 'ANXA5', 'AP2M1', 'AP2S1', 'AP3S1', 'AP4B1', 'APAF1', 'APBB2', 'APEX1', 'APH1A', 'APOBEC3F', 'APOC1', 'APOC2

## kfold 적용하기

In [17]:

skfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

all_x_trains = []
all_y_trains = []
all_x_vals = []
all_y_vals = []

for train_idx, val_idx in skfold.split(fs_train, list(y_subclass)):
    tmp_x_train, tmp_x_val = fs_train.iloc[train_idx], fs_train.iloc[val_idx]
    tmp_y_train, tmp_y_val = y_subclass[train_idx], y_subclass[val_idx]
    all_x_trains.append(tmp_x_train)
    all_y_trains.append(list(tmp_y_train))
    all_x_vals.append(tmp_x_val)
    all_y_vals.append(list(tmp_y_val))

## Model 정의하기

In [7]:
xgb = XGBClassifier(  random_state=42,use_label_encoder=False,eval_metric='mlogloss',tree_method='hist',device ='cuda'  )
lgb = LGBMClassifier(random_state = 42 ,verbose = - 1)
cat = CatBoostClassifier(learning_rate =0.05,random_state=42,loss_function='MultiClass', verbose = 2, task_type='GPU')
bag = BaggingClassifier(random_state =42 )
dtc = DecisionTreeClassifier(random_state=42)
rfc = RandomForestClassifier(random_state=42)
gbc = GradientBoostingClassifier(random_state= 42)
knn = KNeighborsClassifier() 
# models for voting classifier
# Voting Classifier 정의
models = [('xgb', xgb),('lgb', lgb),('bag', bag),('dtc', dtc),('rfc', rfc),('gbc', gbc),('knn', knn)]
models2= [('xgb', xgb),('lgb', lgb),('bag', bag)]
vtc = VotingClassifier(estimators=models, voting='hard',weights=[3,1,1,1,1,2,1])

## 파라미터 탐색

In [8]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import SuccessiveHalvingPruner
from sklearn.metrics import (    accuracy_score,    confusion_matrix,f1_score,precision_score, recall_score,roc_auc_score)

In [ ]:
# [I 2024-09-07 15:32:50,601] A new study created in memory with name: no-name-e60959bc-02db-433e-8223-93507f06f415
# [I 2024-09-07 15:38:01,917] Trial 0 finished with value: 0.2323188089005682 and parameters: {'max_depth': 4, 'learning_rate': 0.07572330688748365, 'n_estimators': 1900, 'colsample_bytree': 0.7675793243106275, 'colsample_bylevel': 0.6047570707738957, 'colsample_bynode': 0.49149473615264755, 'reg_alpha': 0.345191215678889, 'reg_lambda': 0.41674884880509966, 'gamma': 0.2554609703681004, 'min_child_weight': 12, 'eta': 0.012825828583845986}. Best is trial 0 with value: 0.2323188089005682.
# [I 2024-09-07 15:40:26,442] Trial 1 finished with value: 0.3080118192462323 and parameters: {'max_depth': 10, 'learning_rate': 0.07227494410999141, 'n_estimators': 700, 'colsample_bytree': 0.8487404641582925, 'colsample_bylevel': 0.525635447024138, 'colsample_bynode': 0.8738067365730147, 'reg_alpha': 0.8183886016321489, 'reg_lambda': 1.3250566076612647, 'gamma': 0.9266891339372663, 'min_child_weight': 2, 'eta': 0.010295908032806137}. Best is trial 1 with value: 0.3080118192462323.
# [I 2024-09-07 15:44:08,167] Trial 2 finished with value: 0.18333716043333328 and parameters: {'max_depth': 3, 'learning_rate': 0.04658883805769574, 'n_estimators': 1300, 'colsample_bytree': 0.8466001621588627, 'colsample_bylevel': 0.6843525260189696, 'colsample_bynode': 0.5547830774937004, 'reg_alpha': 0.9958534634072155, 'reg_lambda': 3.8449532480892756, 'gamma': 0.43624790131316343, 'min_child_weight': 19, 'eta': 0.01259907477312269}. Best is trial 1 with value: 0.3080118192462323.
# [I 2024-09-07 15:52:00,985] Trial 3 finished with value: 0.18013980379881234 and parameters: {'max_depth': 12, 'learning_rate': 0.05299969500627983, 'n_estimators': 2900, 'colsample_bytree': 0.8591920121863919, 'colsample_bylevel': 0.7615250978671599, 'colsample_bynode': 0.826566778710867, 'reg_alpha': 0.9869729865150384, 'reg_lambda': 3.273259904809398, 'gamma': 0.6906546472122288, 'min_child_weight': 19, 'eta': 0.007282178030691785}. Best is trial 1 with value: 0.3080118192462323.
# [I 2024-09-07 15:55:57,337] Trial 4 finished with value: 0.22945201658804645 and parameters: {'max_depth': 8, 'learning_rate': 0.07012115628986199, 'n_estimators': 1100, 'colsample_bytree': 0.9700887964094991, 'colsample_bylevel': 0.6162227742998847, 'colsample_bynode': 0.964222538495513, 'reg_alpha': 0.6184857134579598, 'reg_lambda': 6.746149050829709, 'gamma': 0.09795886922658582, 'min_child_weight': 15, 'eta': 0.009143417869551989}. Best is trial 1 with value: 0.3080118192462323.
# [I 2024-09-07 16:03:41,416] Trial 5 finished with value: 0.32665294201532613 and parameters: {'max_depth': 4, 'learning_rate': 0.04417028170586605, 'n_estimators': 2500, 'colsample_bytree': 0.8607939202959518, 'colsample_bylevel': 0.5476194858789869, 'colsample_bynode': 0.6184964615455255, 'reg_alpha': 0.386040670187326, 'reg_lambda': 1.6032124420670557, 'gamma': 0.3633314518552518, 'min_child_weight': 2, 'eta': 0.009821861629611444}. Best is trial 5 with value: 0.32665294201532613.
# [I 2024-09-07 16:09:57,600] Trial 6 finished with value: 0.2297013297443012 and parameters: {'max_depth': 9, 'learning_rate': 0.018190352970063137, 'n_estimators': 1300, 'colsample_bytree': 0.7198171941738848, 'colsample_bylevel': 0.7108514325742448, 'colsample_bynode': 0.8802799901389866, 'reg_alpha': 0.2673434973989054, 'reg_lambda': 9.685429633424537, 'gamma': 0.03744454878011597, 'min_child_weight': 14, 'eta': 0.0071208439500681404}. Best is trial 5 with value: 0.32665294201532613.
# [I 2024-09-07 16:18:34,732] Trial 7 finished with value: 0.24208110257390258 and parameters: {'max_depth': 6, 'learning_rate': 0.02962913364531832, 'n_estimators': 2700, 'colsample_bytree': 0.9184871059217495, 'colsample_bylevel': 0.9455103140276656, 'colsample_bynode': 0.5050466193407668, 'reg_alpha': 0.08268902906538023, 'reg_lambda': 3.578858758268957, 'gamma': 0.7425898139579761, 'min_child_weight': 8, 'eta': 0.010183139544087168}. Best is trial 5 with value: 0.32665294201532613.
# [I 2024-09-07 16:21:29,791] Trial 8 finished with value: 0.24697647169231662 and parameters: {'max_depth': 12, 'learning_rate': 0.09313393495371669, 'n_estimators': 900, 'colsample_bytree': 0.7890919260196387, 'colsample_bylevel': 0.4871188801776773, 'colsample_bynode': 0.7761761406047687, 'reg_alpha': 0.7730758640137267, 'reg_lambda': 5.974801413367569, 'gamma': 0.34412093511338093, 'min_child_weight': 7, 'eta': 0.009700812227573914}. Best is trial 5 with value: 0.32665294201532613.
# [I 2024-09-07 16:30:59,926] Trial 9 finished with value: 0.218286720402555 and parameters: {'max_depth': 10, 'learning_rate': 0.00512916907079655, 'n_estimators': 1500, 'colsample_bytree': 0.7031869436029532, 'colsample_bylevel': 0.8938223607794387, 'colsample_bynode': 0.8969178655573269, 'reg_alpha': 0.1413862104280028, 'reg_lambda': 4.155808780113928, 'gamma': 0.1551469201058662, 'min_child_weight': 12, 'eta': 0.00982598510004938}. Best is trial 5 with value: 0.32665294201532613.
# [I 2024-09-07 16:38:10,652] Trial 10 finished with value: 0.31707555746044735 and parameters: {'max_depth': 5, 'learning_rate': 0.036287332894706335, 'n_estimators': 2300, 'colsample_bytree': 0.9880412670899539, 'colsample_bylevel': 0.40239612803959446, 'colsample_bynode': 0.6516806627389311, 'reg_alpha': 0.4409098436775979, 'reg_lambda': 1.98625742819038, 'gamma': 0.5934275937320107, 'min_child_weight': 2, 'eta': 0.011508806145740414}. Best is trial 5 with value: 0.32665294201532613.
# [I 2024-09-07 16:45:22,135] Trial 11 finished with value: 0.3321255129885369 and parameters: {'max_depth': 5, 'learning_rate': 0.040000451307232086, 'n_estimators': 2300, 'colsample_bytree': 0.9712611275256556, 'colsample_bylevel': 0.4266615890819488, 'colsample_bynode': 0.6392944928062945, 'reg_alpha': 0.45531925241423443, 'reg_lambda': 1.7894898246900737, 'gamma': 0.5727663204936421, 'min_child_weight': 1, 'eta': 0.011326667161418913}. Best is trial 11 with value: 0.3321255129885369.
# [I 2024-09-07 16:51:34,852] Trial 12 finished with value: 0.2558115510407185 and parameters: {'max_depth': 6, 'learning_rate': 0.055543646158717895, 'n_estimators': 2100, 'colsample_bytree': 0.9255162972178231, 'colsample_bylevel': 0.4039150721396103, 'colsample_bynode': 0.6454814266746852, 'reg_alpha': 0.5521077753584684, 'reg_lambda': 0.2232317024840098, 'gamma': 0.490779609360758, 'min_child_weight': 5, 'eta': 0.011257950672140584}. Best is trial 11 with value: 0.3321255129885369.
# [W 2024-09-07 16:52:35,907] Trial 13 failed with parameters: {'max_depth': 3, 'learning_rate': 0.03206133219425543, 'n_estimators': 2500, 'colsample_bytree': 0.9149764897836227, 'colsample_bylevel': 0.5106607521251888, 'colsample_bynode': 0.4059191546581141, 'reg_alpha': 0.27385681876871687, 'reg_lambda': 2.28588492643522, 'gamma': 0.3390530526256998, 'min_child_weight': 1, 'eta': 0.008538447759376514} 

In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth',3,12),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1), 
        'n_estimators': trial.suggest_int("n_estimators", 300, 3100, 200),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.01, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'eta': trial.suggest_float('eta', 0.007, 0.013),
        'objective': 'multi:softmax',  # 또는 'multi:softprob'
        'num_class': 26,  # 클래스 수를 지정
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'mlogloss'
    }
    
    score = []
    for all_x_train, all_y_train, all_x_val, all_y_val in zip(all_x_trains, all_y_trains, all_x_vals, all_y_vals):
        clf = XGBClassifier(**params)
        clf.fit(all_x_train, all_y_train)
        
        y_pred = clf.predict(all_x_val)
        y_true = all_y_val
        score.append(f1_score(y_true, y_pred, average='macro'))  # macro F1 score 사용
    score = np.mean(score)
    return score

# Hyperparameter Tuning
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=107), pruner=SuccessiveHalvingPruner())
study.optimize(objective, n_trials=66)

[I 2024-09-07 17:02:03,045] A new study created in memory with name: no-name-f88895d7-b7d1-4ffb-9a11-7d0421bc3bdf
[W 2024-09-07 17:03:11,785] Trial 0 failed with parameters: {'max_depth': 4, 'learning_rate': 0.07572330688748365, 'n_estimators': 1900, 'colsample_bytree': 0.7675793243106275, 'colsample_bylevel': 0.6047570707738957, 'colsample_bynode': 0.49149473615264755, 'reg_alpha': 0.345191215678889, 'reg_lambda': 0.41674884880509966, 'gamma': 0.2554609703681004, 'min_child_weight': 12, 'eta': 0.012825828583845986} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_36/1826835632.py", line 26, in objective
    clf.fit(all_x_train, all_y_train)
  File "/opt/conda/lib/python3.10/site-packages/xgboost/core.py", line 730, in inner_f
    return func(**kwargs)
  File "/opt/conda/lib/python3.10/site-pa

In [15]:
params = {'max_depth': 5, 'learning_rate': 0.040000451307232086, 'n_estimators': 2300, 'colsample_bytree': 0.9712611275256556, 'colsample_bylevel': 0.4266615890819488, 'colsample_bynode': 0.6392944928062945, 'reg_alpha': 0.45531925241423443, 'reg_lambda': 1.7894898246900737, 'gamma': 0.5727663204936421, 'min_child_weight': 1, 'eta': 0.011326667161418913}
xgb = XGBClassifier(**params, 
                    objective='multi:softmax', 
                    num_class=26, 
                    random_state=42, 
                    tree_method='hist', 
                    device='cuda', 
                    eval_metric='mlogloss')

xgb.fit(x_train, y_subclass)
predicts = xgb.predict(x_test) 

In [16]:
original_labels = le_subclass.inverse_transform(predicts)
submisson = pd.read_csv("/kaggle/input/life-resorce/sample_submission.csv")
submisson["SUBCLASS"] = original_labels
submisson.to_csv('./xgb_opt.csv', encoding='UTF-8-sig', index=False)